# Interactive inference tracer

Type a **user query**; see **individually what happens** — what goes through what (embed -> retrieve -> select -> LM alone vs LM+graph). Real 4-bit LM (~2GB, fits 6GB), LM frozen, knowledge in the graph. Run cells 1-2, then use the box in cell 3.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from v5.runtime.membrane import TRMRetriever, learn_any, seed_graph, encode_batch
from v5.runtime.dcpd_latent import WhiteBox
import numpy as np, pandas as pd
from IPython.display import display, Markdown

wb = WhiteBox("Qwen/Qwen2.5-3B-Instruct", quant="4bit")
g  = seed_graph(); retr = TRMRetriever(g)
def clean(t):
    for c in ("
Human:", "Human:", "
You are", "

", "<|"):
        i = t.find(c); t = t[:i] if i > 0 else t
    return t.strip()
for f, name in [("The Klarn Protocol requires three handshake phases: greet, verify, and seal.", "fact_klarn"),
                ("Zephyrite melts at 812 degrees and conducts electricity only when wet.", "fact_zephyrite"),
                ("The Vora index measures how fast a market recovers after a shock, from 0 to 100.", "fact_vora")]:
    learn_any(g, retr, f, name=name)
display(Markdown(f"**ready** - LM {wb.quant} {wb.vram_gb:.2f}GB - graph {len(g)} nodes {g.census()}"))

The tracer: each stage of one inference, shown separately.

In [ ]:
def trace(query):
    display(Markdown(f"## what happens for: *{query}*"))
    qv = encode_batch([query])[0]
    display(Markdown(f"**1 EMBED** - MiniLM -> 384-d vector (norm = {np.linalg.norm(qv):.2f})"))
    M, order = g.matrix(); sims = (M @ qv).tolist()
    rank = sorted(zip(order, sims), key=lambda z: -z[1])[:6]
    df = pd.DataFrame(rank, columns=["node", "similarity"]); df["type"] = [g.get(n).kind for n, _ in rank]
    display(Markdown("**2 RETRIEVE** - neural ranking over the graph (what surfaced):")); display(df)
    top, ts = rank[0]; node = g.get(top); content = node.code or node.description
    display(Markdown(f"**3 SELECT** - `{top}` (sim {ts:.2f}, type {node.kind}):

> {content}"))
    base = clean(wb.generate_plain(f"Answer briefly. {query}", max_new=48))
    display(Markdown(f"**4 LM alone (no memory)** -> {base}"))
    grounded = clean(wb.generate_plain(f"Use ONLY this to answer.
{content}
Question: {query}
Answer:", max_new=48))
    display(Markdown(f"**5 LM + graph (grounded in {top})** -> {grounded}"))
    display(Markdown(f"---
**flow:** query -> MiniLM embed -> graph retrieve ({top}, sim {ts:.2f}) -> LM grounded. "
                     f"Memory turned a guess into {top}'s fact."))

## Type a query and press trace
(edit the seeded facts in cell 1, or `learn_any(g, retr, "...", name="...")` to teach more)

In [ ]:
try:
    import ipywidgets as W
    qbox = W.Text(placeholder="ask anything (e.g. what is the Vora index?)", layout=W.Layout(width="70%"))
    go   = W.Button(description="trace", button_style="primary"); out = W.Output()
    def _run(_):
        out.clear_output()
        with out: trace(qbox.value or "what is the Vora index?")
    go.on_click(_run); display(W.HBox([qbox, go]), out)
except ImportError:
    trace("what is the Vora index?")   # no ipywidgets -> call trace("your query") directly